# Data Persistence



## from events to durable data

- An ingestion tool is responsible to collect events from a source
- A streaming tool is responsible to make the events available in real-time
- A processing tool is responsible to process the events transforming/enriching/aggregating them

:::{.fragment .callout-caution}
All these tools are designed to handle events in memory or for limited time periods, but what happens when the events need to be stored for later use? This is where data persistence comes into play.
:::


## A brief history of data persistence

- **1956 — Direct-access storage:** IBM RAMAC brought random-access disk storage to business computing, making stored data much easier to retrieve and update than punched-card workflows. ([IBM](https://www.ibm.com/history/ramac))

- **1970 — Relational databases:** Codd proposed organizing data as relations, reducing the need for users to know its physical layout. ([Codd, 1970](https://doi.org/10.1145/362384.362685))

- **1987 — Persistence as a research topic:** Atkinson and Buneman surveyed how programming languages could integrate with databases and support persistent data. ([ACM survey](https://doi.org/10.1145/62070.45066))

- **2003 — Distributed files:** Google File System was designed to store and serve large data sets across clusters of commodity machines. ([Google Research](https://research.google/pubs/the-google-file-system/))

- **Today — Managed tables over files:** Table formats such as Apache Iceberg track table metadata and snapshots across data files, adding a table abstraction over distributed storage. ([Apache Iceberg](https://iceberg.apache.org/docs/latest/))

The focus has shifted from keeping data after a program ends to managing large, distributed collections of files as usable tables.

## Three different layers

| Layer | Answers the question | Examples |
|---|---|---|
| **File format** | How are records and values encoded in a file? | Parquet, Avro, ORC, JSON, CSV |
| **Table format** | Which files make up a table, and what is its current schema/state? | Apache Iceberg, Delta Lake, Apache Hudi |
| **Storage** | Where do the bytes live, and how can workers access them? | HDFS, MinIO/S3, cloud object storage, local disk |

Example: a Parquet data file can belong to an Iceberg table stored in MinIO. Spark and DuckDB are **query engines**, not storage layers or file formats.

# File formats

## First: file formats

A file format defines the physical encoding of records and values. CSV and JSON are easy to inspect, but are text formats with limited type information and higher parsing cost.

Analytical pipelines commonly use **columnar** formats (Parquet, ORC) for scans and **row-oriented** formats (Avro) for record-at-a-time exchange. The choice depends on the workload; a format by itself does not provide a database table or a storage service.

## Parquet: columnar analytics

Parquet stores data by column within row groups. A query that needs only `country` and `revenue` can avoid reading unrelated columns. Encodings, compression, and per-column statistics can reduce I/O and help engines skip irrelevant row groups.

```text
row group 0: country block | revenue block | timestamp block | ...
row group 1: country block | revenue block | timestamp block | ...
```

Parquet is a strong default for analytical data files. It does not tell us which files belong to a changing table.

## Avro: row-oriented serialization

Avro stores records in a compact binary form and uses an explicit schema. Its object-container files store the writer schema with the data, making Avro useful for record exchange and event/streaming pipelines. ([Apache Avro documentation](https://avro.apache.org/docs/1.12.0/))

A small Avro record schema looks like this:

```json
{
  "type": "record",
  "name": "ServiceRequest",
  "fields": [
    {"name": "agency", "type": "string"},
    {"name": "borough", "type": ["null", "string"], "default": null}
  ]
}
```

In our Kafka demo, Confluent-framed Avro records carry a **schema ID**; the writer schema is retrieved from Schema Registry.

## ORC: columnar analytics

ORC stores values by column in stripes and includes indexes/statistics that help query engines skip data. It is designed for analytical scans, especially when a query reads only some columns. ([Apache ORC](https://orc.apache.org/))

For the same service-request records, ORC groups values by field rather than placing each complete record next to the next one:

```text
stripe: agency [..] | complaint_type [..] | borough [..] | latitude [..]
```

The practical comparison below writes the same typed columns as Avro, Parquet, and ORC, then compares their sizes and reads back the same summaries.

## Should we consider Apache Arrow?

Yes, as a companion format. Arrow is primarily a **columnar in-memory representation**; Arrow also defines IPC stream and file formats for serializing record batches. This comparison includes an uncompressed Arrow IPC file as a reference point, but it should not be confused with a distributed table format. ([Arrow IPC documentation](https://arrow.apache.org/docs/python/ipc.html))

## Practical: compare file encodings on open data

The example uses a 50,000-row sample from the [NYC 311 Service Requests open dataset](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9). It writes the same selected columns to CSV, Avro, Parquet, ORC, and Arrow IPC, then compares file size, row count, borough counts, and latitude statistics.

The runnable source is [`compare_file_formats.py`](../python/bin/compare_file_formats.py); the snippets below show its key steps. The baseline uses no compression in the binary formats; compression choices can be compared separately.

```bash
cd tap2027
docker build --tag tap:python ./python
mkdir -p python/data
docker run --rm \
  -v "$(pwd)/python/bin:/usr/src/app/bin" \
  -v "$(pwd)/python/data:/usr/src/app/data" \
  -e PYTHON_APP=compare_file_formats.py tap:python
```

The script writes its CSV and encoded files to `tap2027/python/data/formats-demo/`.

## Download and encode the sample

```python
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlretrieve

import pyarrow as pa
import pyarrow.csv as pacsv
from pyarrow import orc
from fastavro import writer, reader

work = Path("formats-demo")
work.mkdir(exist_ok=True)
csv_path = work / "nyc311.csv"
avro_path = work / "nyc311.avro"
orc_path = work / "nyc311.orc"
arrow_path = work / "nyc311.arrow"

columns = {
    "unique_key": pa.string(),
    "created_date": pa.string(),
    "agency": pa.string(),
    "complaint_type": pa.string(),
    "borough": pa.string(),
    "latitude": pa.float64(),
    "longitude": pa.float64(),
}
params = {
    "$select": ",".join(columns),
    "$where": "created_date >= '2024-01-01T00:00:00' AND created_date < '2024-02-01T00:00:00'",
    "$order": "created_date DESC, unique_key DESC",
    "$limit": "50000",
}
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.csv?" + urlencode(params)
urlretrieve(url, csv_path)

table = pacsv.read_csv(
    csv_path,
    convert_options=pacsv.ConvertOptions(
        column_types=columns,
        strings_can_be_null=True,
    ),
)
records = table.to_pylist()

avro_schema = {
    "type": "record",
    "name": "NYC311Request",
    "fields": [
        {"name": "unique_key", "type": "string"},
        {"name": "created_date", "type": ["null", "string"], "default": None},
        {"name": "agency", "type": ["null", "string"], "default": None},
        {"name": "complaint_type", "type": ["null", "string"], "default": None},
        {"name": "borough", "type": ["null", "string"], "default": None},
        {"name": "latitude", "type": ["null", "double"], "default": None},
        {"name": "longitude", "type": ["null", "double"], "default": None},
    ],
}
with avro_path.open("wb") as stream:
    writer(stream, avro_schema, records, codec="null")

orc.write_table(table, orc_path, compression="uncompressed")
with pa.OSFile(str(arrow_path), "wb") as sink:
    with pa.ipc.new_file(sink, table.schema) as arrow_writer:
        arrow_writer.write_table(table)
```

## Compare size and column summaries

The report reads each encoded file back. Equal row counts and matching column summaries confirm that the formats preserve the selected data.

```python
from collections import Counter
from math import isclose
from statistics import fmean

def summarize(rows):
    count = 0
    boroughs = Counter()
    latitudes = []
    for row in rows:
        count += 1
        if row["borough"] is not None:
            boroughs[row["borough"]] += 1
        if row["latitude"] is not None:
            latitudes.append(row["latitude"])
    latitude_stats = (
        (min(latitudes), fmean(latitudes), max(latitudes))
        if latitudes else None
    )
    return count, boroughs.most_common(5), latitude_stats

expected = summarize(records)
def report(label, path, rows):
    result = summarize(rows)
    count, boroughs, latitude_stats = result
    assert count == expected[0]
    assert boroughs == expected[1]
    if latitude_stats is not None:
        assert latitude_stats[0] == expected[2][0]
        assert isclose(latitude_stats[1], expected[2][1], rel_tol=1e-12)
        assert latitude_stats[2] == expected[2][2]
    print(f"{label:9} {path.stat().st_size:>12,} bytes  rows={count:,}")
    print("  borough top 5:", boroughs)
    print("  latitude min/mean/max:", latitude_stats)

report("CSV", csv_path, records)
with avro_path.open("rb") as stream:
    report("Avro", avro_path, reader(stream))
report("ORC", orc_path, orc.read_table(orc_path).to_pylist())
with pa.memory_map(str(arrow_path), "r") as source:
    arrow_table = pa.ipc.open_file(source).read_all()
    report("Arrow IPC", arrow_path, arrow_table.to_pylist())
```

The comparison is illustrative, not a universal ranking: file size depends on schema, null density, data distribution, and compression. Repeat with each format's common codecs, and benchmark column-selective reads separately to see ORC's columnar advantage.

## Row versus column: pizza orders

```text
rows:    [store=1, order=1000, status=accepted] [store=2, order=1001, status=accepted]
columns: store [1, 2] | order [1000, 1001] | status [accepted, accepted]
```

Reading a complete event naturally fits row-oriented serialization; `COUNT(*) GROUP BY store_id` needs only a small subset of columns. Columnar layout allows an engine to avoid reading other columns, but speed depends on data layout, selectivity, compression and engine.

Parquet is binary: use SQL tools or metadata viewers, not a text editor, to inspect it.

## File formats: choose by workload

| Format | Organization | Useful for | Trade-off |
|---|---|---|---|
| CSV | Text, rows | Simple exchange and inspection | Types and nested structures need conventions |
| JSON | Text, records | Readable nested events and APIs | Parsing and repeated field names can be expensive |
| Avro | Binary, rows with schemas | Record exchange, schema-aware events | Not optimized for reading a few columns |
| Parquet / ORC | Binary, columns | Compressed analytical scans | Not human-readable; less suited to single-record edits |
| Arrow IPC | Columnar, file or stream | Fast interchange with Arrow-aware tools | Not a table format; ecosystem and compression differ |

Compression reduces bytes, not necessarily execution time; measure with real data. The same dataset may appear in different formats at different pipeline stages.

## Practical: read Spark's Parquet with DuckDB

First complete the pizza streaming demo and let it write some records. From `quickstart/pizza-kafka-spark`, stop **only** the Spark driver so the exported files are stable, then copy them to an initially empty local directory:

```bash
docker compose stop spark
mkdir pizza-export && docker compose cp spark:/opt/spark/work-dir/pizza-demo/parquet/. ./pizza-export/
```

Create a separate Python environment with `python3 -m venv .duckdb-venv` and `.duckdb-venv/bin/python -m pip install duckdb==2.0.0`. Do not export while the stream is actively writing. Use a new export directory for each copy. These are sample files, not a transactional table snapshot.

## DuckDB: query and inspect the files

Run from the same directory after exporting the pizza data:

```bash
.duckdb-venv/bin/python - <<'PY'
import duckdb
db = duckdb.connect()
path = 'pizza-export/part-*.parquet'
print(db.execute('DESCRIBE SELECT * FROM read_parquet(?)', [path]).fetchall())
print(db.execute('SELECT store_id, COUNT(*) AS orders FROM read_parquet(?) '
                 'GROUP BY store_id ORDER BY store_id', [path]).fetchall())
print(db.execute('SELECT path_in_schema, type, compression '
                 'FROM parquet_metadata(?) LIMIT 10', [path]).fetchall())
PY
```

DuckDB is an embedded, local analytical database. Reading a Parquet glob does **not** make the files an Iceberg table or provide table snapshots.

## Spark SQL: the same pizza data

After exporting the fixed copy in the previous step, read it with Spark SQL too:

```bash
docker run --rm -v "$(pwd)/pizza-export:/data:ro" apache/spark:4.3.0 \
  /opt/spark/bin/spark-sql -e \
  "SELECT store_id, COUNT(*) AS orders FROM parquet.\`/data\` GROUP BY store_id ORDER BY store_id"
```

Compare the per-store counts with DuckDB. Spark uses its file sink's metadata to find committed files; DuckDB reads the copied Parquet files directly. Counts can differ if uncommitted files remain after a failure or interruption: stopping the stream prevents new writes but does not turn a file glob into a transactional snapshot.

# 2. Open table formats

## Why and when did they emerge?

In the late 2010s, data lakes increasingly held many Parquet files on HDFS and object stores. Teams wanted analytics, concurrent ingestion, updates, reliable reads, and multiple engines over the same data.

A plain file directory and traditional partition listings made it difficult to coordinate atomic changes and track exactly which files formed a consistent table. New table formats emerged: **Apache Hudi** grew from Uber's Hoodie work, **Apache Iceberg** from Netflix, and **Delta Lake** from Databricks. They are open projects with distinct metadata protocols.

They build on earlier database and transactional-file-system ideas; this is an evolution, not the invention of transactions.

## Files alone are not a table

A directory of Parquet files does not inherently say which files are current, how a schema changed, or which set of files represented the table at a previous point in time.

A query engine can read files, but concurrent writers, deletes, schema changes, and consistent snapshots require a shared table-level protocol and metadata.

## Next: open table formats

An **open table format** organizes data files into a logical table and maintains metadata describing schemas, partitions, snapshots, and the files in each table state.

It enables engines to coordinate table changes and read a consistent snapshot while leaving the underlying files in a data lake. It is not a storage system: the files and metadata still need to live somewhere accessible.

## Iceberg and Delta Lake

| | Apache Iceberg | Delta Lake |
|---|---|---|
| Role | Open table format | Open table format |
| Typical capabilities | Snapshots, schema/partition evolution, concurrent table operations | Transaction log, snapshots, schema evolution, table operations |
| Data files | Commonly Parquet; also supports Avro and ORC | Commonly Parquet |
| Engines | Broad multi-engine ecosystem | Broad ecosystem, with particularly strong Databricks integration |

Both solve table-management problems over data-lake files. Their metadata layouts and engine integrations differ; choose based on interoperability, platform, and operational requirements.

## Table format vs database

| | Open table format | Database management system |
|---|---|---|
| Main role | Specify table metadata, file membership and commit rules | Run a system that stores, queries and manages data |
| Execution | Requires a compatible engine, catalog and storage | Typically includes query execution and storage management |
| Typical strengths | Shared analytical data and interoperability between engines | Depending on the DBMS: constraints, indexes, fine-grained access, low-latency transactions |
| Examples | Iceberg, Delta Lake | PostgreSQL, DuckDB, ClickHouse |

A format is not a running DBMS; a DBMS can also read or write an open table format. Capabilities and guarantees depend on the specific engine, catalog, and supported operations.

## What does ACID mean here?

- **Atomicity:** a supported table commit is visible as a whole, not as half of a set of new files.
- **Consistency:** committed changes must respect the table format's rules; business invariants and constraints still need explicit enforcement.
- **Isolation:** readers see a committed snapshot while another writer commits; conflict detection and isolation depend on operation and implementation.
- **Durability:** committed metadata and files survive only as reliably as the catalog and underlying storage; plan backups and retention.

Iceberg uses optimistic concurrency and atomic metadata commits. Do **not** infer a cross-table transaction, enforced primary key, or exactly-once Kafka-to-table processing from the word ACID alone.

## Example: an atomic table update

Imagine a committed Iceberg snapshot containing 100 pizza orders. A writer prepares an update that changes 20 rows:

1. It writes new data and metadata files; readers still see the old snapshot.
2. The catalog atomically commits the new metadata reference.
3. A new reader sees the new snapshot; an existing reader can keep using the old one.

If the commit fails, prepared files are not part of the current table. Concurrent writers may retry or fail validation. This is a **conceptual example**; the future lab will test the behavior.

## Different persistence roles: Kafka, Elasticsearch, and a Lakehouse

| System | Typical role | What it is good at |
|---|---|---|
| **Kafka** | Retained event log | Decoupling producers and consumers; replaying events within the configured retention window |
| **Elasticsearch** (later lesson) | Search/serving index | Low-latency text, filter, and aggregation queries over indexed documents |
| **Iceberg on HDFS/object storage** | Analytical table and historical store | SQL analytics over large datasets, table snapshots, and long-term retention policies |

These are common roles, not hard limits: Kafka and Elasticsearch can retain data for a long time, and lakehouse tables can be expired or deleted. **“Long-term” describes the intended retention and analytical role, not an automatic durability guarantee.** Configure retention, replication, backup, and recovery for each system.

## Iceberg snapshots: a table has history

Iceberg commits create new table snapshots. Readers can use a consistent snapshot while a writer commits another change. Schema and partition evolution are tracked in metadata rather than requiring every old file to be rewritten immediately.

Snapshots support inspection and time travel, and old unreferenced files can later be removed through explicit maintenance. A snapshot is metadata about a table state, not a backup or a replication strategy.

## Lakehouse: combine the strengths

A **lakehouse** keeps data in scalable, relatively open data-lake storage while adding table semantics commonly associated with warehouses: consistent reads, transactions, schema management, and SQL analytics.

```text
Spark / Trino / other engines
          | read and write table snapshots
Iceberg or Delta table metadata
          | references
Parquet / Avro / ORC data files
          | stored in
HDFS / MinIO / cloud object storage
```

The lakehouse is an architectural pattern, not a single product.

# 3. Shared and distributed storage

Storage makes the data and metadata accessible; it does not define the table format.

## Finally: storage

Storage determines durability, capacity, access patterns, failure behavior, and how workers find the same data.

- **Local disk**: fast and useful for scratch data, but normally tied to one machine.
- **HDFS**: distributed filesystem; files are split into blocks managed by DataNodes, with namespace/metadata managed by a NameNode.
- **Object storage**: objects in buckets, accessed through an API; examples include MinIO (S3-compatible), Amazon S3, Google Cloud Storage, and Azure Blob Storage.

HDFS and object storage are different systems and APIs. Neither is the table format.

## HDFS recap: shared namespace for a cluster

Recall the HDFS roles from Flume: the NameNode manages filesystem namespace and block locations; DataNodes store blocks. Files are split into blocks and can be replicated across DataNodes for fault tolerance.

From a client or Spark worker, a path such as `hdfs://namenode:9000/warehouse/...` names data in the shared filesystem. A worker does not need the file to exist on its own local disk.

HDFS is a distributed filesystem—not an object store and not a table format.

## Object storage: MinIO and cloud services

MinIO provides an S3-compatible object-store API and can be run on premises for a lab or private environment. Public-cloud alternatives include Amazon S3, Google Cloud Storage, and Azure Blob Storage.

Object stores scale across many clients and are common lakehouse storage targets. They expose object/bucket operations rather than a full POSIX filesystem; clients and table formats must use compatible connectors and commit behavior. Plan credentials, TLS, versioning/retention, and failure recovery deliberately.

Use HDFS if the teaching goal is the familiar Hadoop filesystem; use MinIO to introduce S3-compatible object storage without a cloud account.

## The pieces fit together

```text
Spark SQL
  | Iceberg catalog: table name and current metadata
  v
Iceberg metadata files  --->  snapshots / schemas / manifests
  |
  +---- references Parquet data files
                 |
                 v
          HDFS or MinIO/S3
```

The catalog helps Spark resolve a table and coordinate commits; Iceberg metadata identifies the files in a snapshot; storage makes those files available to the Spark driver and executors.

## Spark SQL + Iceberg: session configuration

This is a **separate Spark 3.5 / Scala 2.12 example**, not the Spark 4.3 pizza deployment. It requires a working HDFS service at `namenode:9000`, network access from driver and executors, warehouse permissions, and the matching Iceberg runtime. The pizza Compose stack does not provide HDFS.

```bash
spark-sql \
  --packages org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1 \
  --conf spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions \
  --conf spark.sql.catalog.demo=org.apache.iceberg.spark.SparkCatalog \
  --conf spark.sql.catalog.demo.type=hadoop \
  --conf spark.sql.catalog.demo.warehouse=hdfs://namenode:9000/warehouse/iceberg
```

Use dependency versions compatible with the Spark/Scala versions in the cluster; do not mix arbitrary runtime JARs. The available Iceberg runtimes must be checked before upgrading the pizza lab; an Iceberg Spark 4.3 runtime was not found in the official Maven repository when preparing this lesson.

## Next lab: MinIO and a shared catalog

To extend the pizza demo, use MinIO as an S3-compatible warehouse and a compatible Iceberg **REST catalog** to coordinate the current table metadata and atomic commits. Both Spark workers and the catalog need appropriate access to the object store; the driver needs access to the catalog.

This is a **future lab**, not a configuration for the current pizza stack. Verify catalog commit semantics and engine/version compatibility; use the appropriate S3 FileIO and AWS bundle. Store credentials outside notebooks and source control. A single MinIO container is shared storage, but not a fault-tolerant distributed deployment.

## Create and query a table

Run the statements in the configured Spark SQL session:

```sql
CREATE NAMESPACE IF NOT EXISTS demo.course;

CREATE TABLE IF NOT EXISTS demo.course.events (
  event_id BIGINT,
  event_type STRING,
  source STRING
) USING iceberg;

INSERT INTO demo.course.events VALUES
  (1, 'login', 'web'),
  (2, 'purchase', 'mobile'),
  (3, 'logout', 'web');

SELECT event_type, COUNT(*) AS events
FROM demo.course.events
GROUP BY event_type;
```

Spark SQL writes Iceberg table data and metadata to the configured warehouse, not to an implicit local table directory.

## Inspect snapshots and time travel

Iceberg exposes metadata tables that help explain the table's history:

```sql
SELECT committed_at, snapshot_id, operation
FROM demo.course.events.snapshots
ORDER BY committed_at;
```

Record the ID of the first snapshot, then insert another row. Use the saved ID to read the earlier state:

```sql
INSERT INTO demo.course.events VALUES (4, 'purchase', 'web');
SELECT * FROM demo.course.events ORDER BY event_id;
SELECT * FROM demo.course.events VERSION AS OF <snapshot_id>;
```

Replace `<snapshot_id>` with the generated numeric ID, then compare this result with `SELECT * FROM demo.course.events`. Snapshot IDs are generated values—never assume a particular ID.

## Schema evolution: a table operation

Continue the same Spark SQL session after creating `demo.course.events`:

```sql
ALTER TABLE demo.course.events ADD COLUMN amount DOUBLE;
INSERT INTO demo.course.events (event_id, event_type, source, amount)
VALUES (5, 'purchase', 'web', 19.5);
SELECT event_id, amount FROM demo.course.events ORDER BY event_id;
```

Earlier rows have `NULL` for the newly added column; old files need not be rewritten just to add it. Schema evolution is different from registering an Avro schema for Kafka messages.

## DuckDB and Iceberg

DuckDB can query an Iceberg table through its Iceberg extension. In DuckDB 2.0, direct `iceberg_scan` of a table's metadata is **read-only**; a supported REST catalog enables additional operations, including writes. Engine/version and table-feature compatibility still matter.

```sql
INSTALL iceberg;
LOAD iceberg;
SELECT * FROM iceberg_scan('<path-to-current-table-metadata.json>');
```

This is an illustrative template: replace the path with accessible metadata and configure object-store access if necessary. Do not query Iceberg data files with a Parquet glob: that ignores table snapshots and delete files.

## Proposed lab: observe atomicity and isolation

In the future MinIO/Iceberg pizza lab, first write a known baseline and record snapshot A. Prepare a large update while a second session queries the table repeatedly.

- Before the commit, the reader sees A; after commit, new queries see B. Verify counts and affected IDs against both snapshots, not only the UI.
- For a controlled failure, stop the writer **before commit**, confirm the current snapshot and results remain A, then inspect any uncommitted files separately.
- Finally, retry/commit successfully and compare A with B using time travel. For concurrent conflicting writes, inspect whether the catalog retries or rejects the operation.

Use an isolated test table and a catalog with verified atomic commits. A reliable pre-commit pause or failure needs a controlled test hook, not just timing. This demonstrates table-level atomic visibility and snapshot isolation; it does not by itself prove every ACID property or exactly-once ingestion.

## The two-worker experiment: why shared storage matters

Run a Spark standalone cluster with one driver and **two workers**. First point the Iceberg warehouse at a path local to one container, for example `file:///tmp/warehouse`.

A path on worker 1's local filesystem is not automatically visible on worker 2. Executors scheduled on worker 2 cannot reliably read files that exist only on worker 1; identically named local paths on different containers may even refer to different bytes.

Repeat with the warehouse on a shared filesystem, such as `hdfs://namenode:9000/warehouse/iceberg`, or a correctly configured shared object store. Both workers then resolve the same table metadata and data-file locations.

## Observe the experiment

1. Create a table and insert enough partitioned data to generate work for multiple tasks.
2. Query the table from Spark SQL and inspect the Spark UI **Executors** and **Stages** views; verify that tasks ran on both workers.
3. Compare a local warehouse with HDFS/MinIO as the warehouse, checking file visibility and query results from each worker.
4. Stop the worker that created local files and discuss which table files the remaining worker can still read.

A tiny query may run on only one worker, so use enough input partitions and inspect task placement. Distributed compute does not make node-local files distributed or shared.

## Takeaways

- **Parquet / Avro / ORC** describe the encoding of files.
- **Iceberg / Delta Lake** describe table state and coordinate table changes over files.
- **HDFS / MinIO / cloud object storage** provide shared places for those files and metadata.
- Spark can distribute computation across workers, but the table must be stored where every worker that needs it can access it.

### References

- [Apache Iceberg documentation](https://iceberg.apache.org/docs/latest/)
- [Apache Spark SQL guide](https://spark.apache.org/docs/latest/sql-programming-guide.html)
- [Apache Parquet](https://parquet.apache.org/docs/)
- [Apache Avro](https://avro.apache.org/docs/)
- [MinIO documentation](https://min.io/docs/)